# A RETRIEVER searches indexed data and returns relevant nodes. It does not normally generate the final natural-language answer.

In [1]:
import logging
import warnings

warnings.filterwarnings("ignore", message="Mixing V1 models and V2 models")
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("httpcore").setLevel(logging.WARNING)
logging.getLogger("llama_index").setLevel(logging.ERROR)
logging.disable(logging.WARNING)

In [2]:
from llama_index.core import SimpleDirectoryReader
from dotenv import load_dotenv

load_dotenv()

# Load PDF
documents = SimpleDirectoryReader(input_files=["data/HopeAI.pdf"]).load_data()

<h1 style="color: #7C3AED;">1. VECTOR INDEX RETRIEVER</h1>

# Semantic search using embeddings

<div align="center">
    <img src="images/vectorindex.png" width="300" height="700">
</div>

In [3]:
from llama_index.core import VectorStoreIndex

# Create index
index = VectorStoreIndex.from_documents(documents)

# Create retriever
retriever = index.as_retriever(similarity_top_k=3)

# User question
question = "What courses are offered by Hope AI?"

# Retrieve relevant nodes
nodes = retriever.retrieve(question)

# Generate answer
query_engine = index.as_query_engine(similarity_top_k=3)
response = query_engine.query(question)
print(response)

Hope AI offers courses in Artificial Intelligence and Data Science.


<h1 style="color: #7C3AED;">2. BM25 RETRIEVER</h1>

# Keyword-based lexical search

<div align="center">
    <img src="images/bm25.png" width="300" height="700">
</div>

In [4]:
from llama_index.core import VectorStoreIndex
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.core import get_response_synthesizer

# Create an index only to obtain nodes
index = VectorStoreIndex.from_documents(documents)
nodes = list(index.docstore.docs.values())

# Create BM25 retriever
retriever = BM25Retriever.from_defaults(nodes=nodes, similarity_top_k=3)

question = "What courses are offered by Hope AI?"

# Retrieve nodes
retrieved_nodes = retriever.retrieve(question)

for node in retrieved_nodes:
    print(node.text[:500])
    print("-" * 50)

response_synthesizer = get_response_synthesizer()
response = response_synthesizer.synthesize(question, nodes=retrieved_nodes)
print(response)

Welcome to Hope AI, an E-Learning platform committed to providing world-class Artificial 
Intelligence and Data Science courses. Our goal is to empower individuals passionate about 
advancing their knowledge in these fields. Our courses offer comprehensive knowledge and practical 
skills in AI and Data Science, supported by mentorship from industry experts. 
 
Founded in 2019, Hope AI started as a startup under the MSME scheme. Since then, we've been 
dedicated to offering quality education and 
--------------------------------------------------
- Classification, Regression, Clustering, Deep Learning, Time Series Analysis, NLP 
 
Cloud and Devops  
1) AWS 
2) GCP 
3) Azure 
 
Advanced AI Topics 
Transformer Architecture  
LLM 
AI Agents 
RAG 
 
These topics are covered with real time applications deployment.  
 
 
Life Time course access.
--------------------------------------------------
- Flexibility: Fits busy schedules 
  - Personalized Mentorship: Tailored guidance for each studen

<h1 style="color: #7C3AED;">3. KEYWORD TABLE SIMPLE RETRIEVER</h1>

# Retrieves nodes using extracted keywords

<div align="center">
    <img src="images/keywordtable.png" width="300" height="700">
</div>

In [5]:
from llama_index.core import KeywordTableIndex

index = KeywordTableIndex.from_documents(documents)

retriever = index.as_retriever(retriever_mode="simple", similarity_top_k=3)

question = "What courses are offered by Hope AI?"

nodes = retriever.retrieve(question)
response = index.as_query_engine().query(question)
print(response)

Python Basics, Libraries in Python, Data Science, Univariate Analysis, Bivariate Analysis, Machine Learning, Advanced ML, Artificial Neural Network, Convolutional Neural Network, RNN, LSTMs, Deep Learning & Computer Vision, Time Series Analysis & Forecasting, Natural Language Processing, Web Development with Django and AI Model Integration.


<h1 style="color: #7C3AED;">4. SUMMARY INDEX RETRIEVER</h1>

# Retrieves nodes from a summary index

<div align="center">
    <img src="images/summaryindex.png" width="300" height="700">
</div>

In [6]:
from llama_index.core import SummaryIndex

index = SummaryIndex.from_documents(documents)

retriever = index.as_retriever()

question = "Give me a summary of Hope AI."

nodes = retriever.retrieve(question)
response = index.as_query_engine().query(question)
print(response)

Hope AI is an E-Learning platform that offers comprehensive courses in Artificial Intelligence and Data Science. Founded in 2019 as a startup under the MSME scheme, it has since become a private limited company. The platform provides self-paced learning in AI and Data Science, with personalized mentorship and support available 24/7. The courses cover a wide range of topics including Python basics, Data Science, Machine Learning, Deep Learning, Natural Language Processing, and more. Hope AI also offers placement support, lifetime course access, and real-time scenario-based capstone projects to help students excel in their careers.


<h1 style="color: #7C3AED;">5. QUERY FUSION RETRIEVER</h1>

# Combines results from multiple retrievers

<div align="center">
    <img src="images/queryfusion.png" width="300" height="700">
</div>

In [7]:
from llama_index.core import VectorStoreIndex
from llama_index.core.retrievers import QueryFusionRetriever

index = VectorStoreIndex.from_documents(documents)

vector_retriever = index.as_retriever(similarity_top_k=3)

retriever = QueryFusionRetriever([vector_retriever], num_queries=3, use_async=False)

question = "What training programs are available at Hope AI?"

nodes = retriever.retrieve(question)
query_engine = index.as_query_engine()
response = query_engine.query(question)
print(response)

Hope AI offers training programs in Artificial Intelligence and Data Science. The courses cover a wide range of topics such as Python, Machine Learning, NLP, Time Series Analysis, Deep Learning, and more. Additionally, there are specialized programs on Cloud and DevOps platforms like AWS, GCP, and Azure. The curriculum also includes advanced AI topics like Transformer Architecture, LLM, AI Agents, and RAG, with a focus on real-time applications deployment.


<h1 style="color: #7C3AED;">6. NLSQL RETRIEVER</h1>

# Converts natural language into SQL

<div align="center">
    <img src="images/nlsql.png" width="300" height="700">
</div>

In [8]:
from llama_index.core import SQLDatabase
from llama_index.core.retrievers import NLSQLRetriever
from sqlalchemy import create_engine, text
from llama_index.llms.openai import OpenAI

llm = OpenAI(model="gpt-4o-mini", temperature=0)

# Create database
engine = create_engine("sqlite:///hopeai.db")

# Create table
with engine.connect() as conn:

    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS courses (
            id INTEGER,
            course_name TEXT,
            duration TEXT,
            mode TEXT
        )
    """))

    conn.execute(text("DELETE FROM courses"))

    conn.execute(text("""
        INSERT INTO courses VALUES
        (1, 'AI and Data Science', '6 Months', 'Online'),
        (2, 'Python', '3 Months', 'Online'),
        (3, 'Generative AI', '3 Months', 'Online')
    """))

    conn.commit()

# Connect LlamaIndex
sql_database = SQLDatabase(engine, include_tables=["courses"])

retriever = NLSQLRetriever(sql_database=sql_database, tables=["courses"], llm=llm, return_raw=True)
question = "What courses are available?"
response = query_engine.query(question)
print(response)

Python Basics, Libraries in Python (Numpy, Pandas, Matplotlib, Sklearn, Seaborn), Data Science (Loading Dataset, Missing Data, Categorical Data, Feature Scaling), Univariate Analysis in Data Science, Bivariate Analysis in Data Science, Machine Learning (Regression, Classification, Clustering), Advanced ML (Feature Selection, Dimensionality Reduction), Artificial Neural Network, Convolutional Neural Network, RNN, LSTMs, Deep Learning & Computer Vision, Time Series Analysis & Forecasting, Natural Language Processing (Tokenization, Stemming, Sentiment Analysis, Chatbot Creation), Web Development with Django and AI Model Integration, Cloud and Devops (AWS, GCP, Azure), Advanced AI Topics (Transformer Architecture, LLM, AI Agents, RAG).


<h1 style="color: #7C3AED;">7. RECURSIVE RETRIEVER</h1>

# RecursiveRetriever can follow references from one retrieved object to another.

<div align="center">
    <img src="images/recursive.png" width="300" height="700">
</div>

In [9]:
from llama_index.core import VectorStoreIndex
from llama_index.core.retrievers import RecursiveRetriever
from llama_index.core import get_response_synthesizer

index = VectorStoreIndex.from_documents(documents)

vector_retriever = index.as_retriever(similarity_top_k=3)

retriever_dict = {"vector": vector_retriever}

retriever = RecursiveRetriever("vector", retriever_dict=retriever_dict, verbose=True)

question = "What courses are offered by Hope AI?"
nodes = retriever.retrieve(question)
#for node in nodes:
#    print(node.text[:500])
#    print("-" * 50)
response_synthesizer = get_response_synthesizer()
response = response_synthesizer.synthesize(question, nodes)
print(response)

Retrieving with query id None: What courses are offered by Hope AI?
Retrieving text node: Welcome to Hope AI, an E-Learning platform committed to providing world-class Artificial 
Intelligence and Data Science courses. Our goal is to empower individuals passionate about 
advancing their knowledge in these fields. Our courses offer comprehensive knowledge and practical 
skills in AI and Data Science, supported by mentorship from industry experts. 
 
Founded in 2019, Hope AI started as a startup under the MSME scheme. Since then, we've been 
dedicated to offering quality education and training in AI and Data Science. Proudly, we successfully 
converted to a private limited company in December 2022, demonstrating our commitment to 
providing high-quality education long-term. 
 
Our mission is to empower individuals with the knowledge and skills to succeed in the AI and Data 
Science industry. Through expert mentorship and practical training, we aim to help students achieve 
their goals an

<h1 style="color: #7C3AED;">8. ROUTER RETRIEVER</h1>

# RouterRetriever chooses which retriever should answer the question.

<div align="center">
    <img src="images/routerretriever.png" width="300" height="700">
</div>

In [10]:
from llama_index.core import VectorStoreIndex
from llama_index.core.retrievers import RouterRetriever
from llama_index.core.selectors import PydanticSingleSelector
from llama_index.core.tools import RetrieverTool
from llama_index.core import PromptTemplate, get_response_synthesizer
from llama_index.retrievers.bm25 import BM25Retriever


prompt = PromptTemplate(
    "Context:\n{context_str}\n\n"
    "Question: {query_str}\n\n"
    "Answer only what is asked in the question. "
    "Do not give additional information from document"
)
# Vector index
vector_index = VectorStoreIndex.from_documents(documents)

vector_retriever = vector_index.as_retriever(
    similarity_top_k=1
)

# BM25
nodes = list(vector_index.docstore.docs.values())

bm25_retriever = BM25Retriever.from_defaults(
    nodes=nodes,
    similarity_top_k=1
)

# Create tools
vector_tool = RetrieverTool.from_defaults(
    retriever=vector_retriever,
    description="Use this for semantic meaning based questions."
)

bm25_tool = RetrieverTool.from_defaults(
    retriever=bm25_retriever,
    description="Use this for exact keyword based questions."
)

# Router
retriever = RouterRetriever(
    selector=PydanticSingleSelector.from_defaults(),
    retriever_tools=[
        vector_tool,
        bm25_tool
    ]
)

question = "What all topics covered in Machine Learning?"

nodes = retriever.retrieve(question)

for node in nodes:
    print(node.text[:500])



response_synthesizer = get_response_synthesizer(
    text_qa_template=prompt
)
response = response_synthesizer.synthesize(question, nodes)
print(response)

- Classification, Regression, Clustering, Deep Learning, Time Series Analysis, NLP 
 
Cloud and Devops  
1) AWS 
2) GCP 
3) Azure 
 
Advanced AI Topics 
Transformer Architecture  
LLM 
AI Agents 
RAG 
 
These topics are covered with real time applications deployment.  
 
 
Life Time course access.
Classification, Regression, Clustering, Deep Learning, Time Series Analysis, NLP


<h1 style="color: #7C3AED;">9. TREEALLLEAF RETRIEVER</h1>

# A tree index organizes information hierarchically. A leaf-level retrieval strategy can access the information stored at the bottom of that hierarchy.

<div align="center">
    <img src="images/treeallleaf.png" width="300" height="700">
</div>

In [11]:
from llama_index.core import TreeIndex, get_response_synthesizer
from llama_index.core.query_engine import RetrieverQueryEngine

index = TreeIndex.from_documents(documents)

# CREATE TREE ALL LEAF RETRIEVER
retriever = index.as_retriever(retriever_mode="all_leaf")
print("TreeAllLeafRetriever created")

# RETRIEVE ALL LEAF NODES
nodes = retriever.retrieve("What courses are offered by Hope AI?")
#print("\nNumber of retrieved nodes:", len(nodes))

# DISPLAY RETRIEVED NODES
print("\n" + "=" * 70)
print("RETRIEVED LEAF NODES")
print("=" * 70)

for i, node in enumerate(nodes, start=1):

    print(f"\nLeaf Node {i}")
    print("-" * 50)

    print("Node ID:")
    print(node.node.node_id)

    print("\nText:")
    print(node.node.text[:500])

    print("\nScore:")
    print(node.score)

# RESPONSE SYNTHESIZER
response_synthesizer = get_response_synthesizer(response_mode="compact")

# CREATE QUERY ENGINE
query_engine = RetrieverQueryEngine(retriever=retriever, response_synthesizer=response_synthesizer)

# GENERATE FINAL ANSWER
response = query_engine.query("What courses are offered by Hope AI?")
print(response)

TreeAllLeafRetriever created

RETRIEVED LEAF NODES

Leaf Node 1
--------------------------------------------------
Node ID:
9e5352de-f24f-4301-9ef2-7e0cd8767ec0

Text:
Welcome to Hope AI, an E-Learning platform committed to providing world-class Artificial 
Intelligence and Data Science courses. Our goal is to empower individuals passionate about 
advancing their knowledge in these fields. Our courses offer comprehensive knowledge and practical 
skills in AI and Data Science, supported by mentorship from industry experts. 
 
Founded in 2019, Hope AI started as a startup under the MSME scheme. Since then, we've been 
dedicated to offering quality education and 

Score:
None

Leaf Node 2
--------------------------------------------------
Node ID:
68e86929-9aaf-4976-a2fd-8bba083c8a75

Text:
- Flexibility: Fits busy schedules 
  - Personalized Mentorship: Tailored guidance for each student 
  - Convenience: Learn from home, no travel required 
 
Syllabus Overview: 
  - Python Basics 
  - L